In [ ]:
import os
import subprocess
import glob
import gzip
import shutil
import pyvo as vo

# --- 0. FIX THE SCISERVER HEASOFT ENVIRONMENT ---
my_env = os.environ.copy()
my_env['HEADASNOQUERY'] = '1'

if 'PFILES' in my_env:
    my_env['PFILES'] = my_env['PFILES'].replace('~', '/home/idies')
else:
    headas = my_env.get('HEADAS', '/opt/heasoft/x86_64-pc-linux-gnu-libc2.31')
    my_env['PFILES'] = f"/home/idies/pfiles;{headas}/syspfiles"

os.makedirs('/home/idies/pfiles', exist_ok=True)

# --- 1. Define ALL Targets ---
# This is a LIST of dictionaries
targets = [
    {"name": "AO 0235+164","ra": 39.66208, "dec": 16.61639}
]

username = 'joel756' 

ftp_base_path = '/home/idies/workspace/headata/FTP/swift/data/obs/'
output_dir = f'/home/idies/workspace/Storage/{username}/persistent/AO_0235+164_data/'
os.makedirs(output_dir, exist_ok=True)

source_reg = f'/home/idies/workspace/Storage/{username}/persistent/source.reg'
bkg_reg = f'/home/idies/workspace/Storage/{username}/persistent/bkg.reg'

tap_service = vo.dal.TAPService("https://heasarc.gsfc.nasa.gov/xamin/vo/tap")

# --- 2. The Grand Extraction Loop ---
# FIXED: We now iterate through the list properly!
for target in targets:
    target_name = target["name"]
    #targ_id = target["target_id"]
    ra = target["ra"]
    dec = target["dec"]
    
    print(f"\n=============================================")
    print(f"🚀 INITIATING PIPELINE FOR: {target_name}")
    print(f"=============================================")
    
    with open(source_reg, 'w') as f:
        f.write(f"fk5; circle({ra}, {dec}, 5\")\n")
    with open(bkg_reg, 'w') as f:
        f.write(f"fk5; annulus({ra}, {dec}, 7\", 40\")\n")

    # Query the Swift Master Catalog using PyVO
    query = f"""
        SELECT obsid
        FROM swiftmastr
        WHERE start_time >= 55927 AND start_time <= 61192
        AND CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {ra}, {dec}, 0.05)) = 1
    """
    
    try:
        result_set = tap_service.search(query).to_table()
        obsids = result_set['obsid'].astype(str).tolist()
        print(f"✅ Discovered {len(obsids)} observations via TAP query. Extracting images...")
        
        # NOTE: Your actual uvotimsum and uvotsource extraction loop 
        # (looping over the 'obsids' list) will go right here!
        
    except Exception as e:
        print(f"❌ Failed to query data for {target_name}: {e}")
        continue

        
    processed_count = 0
    
    for obs in obsids:  
        obs = obs.zfill(11)
        
        pattern1 = os.path.join(ftp_base_path, '*', obs, 'uvot', 'image')
        pattern2 = os.path.join(ftp_base_path, obs, 'uvot', 'image')
        
        matched_dirs = glob.glob(pattern1)
        if matched_dirs:
            uvot_dir = matched_dirs.pop(0) 
        elif os.path.exists(pattern2):
            uvot_dir = pattern2
        else:
            continue
        
        for filename in os.listdir(uvot_dir):
            if filename.endswith('sk.img.gz'):
                
                filter_id = filename.split('_').pop(0)
                rel_outfile = f"{target_name}_{obs}_{filter_id}_mag.fits"
                full_outfile_path = os.path.join(output_dir, rel_outfile)
                
                # --- THE SKIP LOGIC ---
                # Check if this exact .fits file already exists in UVOT_Results
                if os.path.exists(full_outfile_path):
                    print(f"  ⏭️ Skipping {rel_outfile} (Already processed)")
                    processed_count += 1
                    continue
                # ----------------------
                
                raw_img = os.path.join(uvot_dir, filename)
                unzipped_filename = "temp_" + filename.replace('.gz', '')
                temp_img_unzipped = os.path.join(output_dir, unzipped_filename)
                
                with gzip.open(raw_img, 'rb') as f_in:
                    with open(temp_img_unzipped, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
                
                rel_infile = unzipped_filename
                rel_summed = f"{target_name}_{obs}_{filter_id}_sum.img"
                
                cmd1 = [
                    'uvotimsum', 
                    f'infile={rel_infile}', 
                    f'outfile={rel_summed}', 
                    'exclude=NONE', 
                    'clobber=yes'
                ]
                
                res1 = subprocess.run(cmd1, env=my_env, cwd=output_dir, capture_output=True, text=True)
                
                if res1.returncode == 0:
                    cmd2 = [
                        'uvotsource', 
                        f'image={rel_summed}', 
                        f'srcreg={source_reg}', 
                        f'bkgreg={bkg_reg}', 
                        f'outfile={rel_outfile}', 
                        'sigma=5.0',
                        'clobber=yes'
                    ]
                    res2 = subprocess.run(cmd2, env=my_env, cwd=output_dir, capture_output=True, text=True)
                    
                    if res2.returncode == 0:
                        processed_count += 1
                        print(f"  ✅ Success! Saved to {rel_outfile}")
                    
                    if os.path.exists(os.path.join(output_dir, rel_summed)):
                        os.remove(os.path.join(output_dir, rel_summed))
                    
                if os.path.exists(temp_img_unzipped):
                    os.remove(temp_img_unzipped)
                    
    print(f"✨ Finished {target_name}! Successfully processed {processed_count} unique filter snapshots.")

print("\n🎉 ALL BLAZARS SUCCESSFULLY EXTRACTED TO PERSISTENT STORAGE!")


🚀 INITIATING PIPELINE FOR: AO 0235+164
✅ Discovered 89 observations via TAP query. Extracting images...
  ✅ Success! Saved to AO 0235+164_00030880103_sw00030880103uvv_mag.fits
  ✅ Success! Saved to AO 0235+164_00030880103_sw00030880103um2_mag.fits
  ✅ Success! Saved to AO 0235+164_00030880103_sw00030880103uw1_mag.fits


In [7]:
import shutil

# Define the folder to zip and the output zip file name
results_dir = '/home/idies/workspace/Storage/joel756/persistent/UVOT_data'
zip_output = '/home/idies/workspace/Storage/joel756/persistent/UVOT_data'

# Create the zip archive
shutil.make_archive(zip_output, 'zip', results_dir)
print("✅ Zipping complete! You can now download UVOT_data")


✅ Zipping complete! You can now download UVOT_data
